In [ ]:
# pip install trl pydantic datasets peft bitsandbytes
# pip install flash-attn --no-build-isolation
# pip install liger-kernel transformers==4.51.3
# pip install unsloth

## Setup

In [2]:

import logging
import math
import os
import torch
from typing import Optional, List, Literal
# from unsloth import FastLanguageModel

# Third-party imports
from datasets import Dataset, load_dataset
from peft import LoraConfig as PeftLoraConfig, get_peft_model, prepare_model_for_kbit_training
from pydantic import BaseModel, Field
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTConfig, SFTTrainer
# For vLLM, you may need to install it separately: pip install vllm
# from vllm import LLM, SamplingParams

# --- Basic Configuration ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

# --------------------------------------------------------------------------
# SECTION 1: CONFIGURATION (Pydantic Models)
# --------------------------------------------------------------------------

class PeftConfig(BaseModel):
    """Configuration for Parameter-Efficient Fine-Tuning (PEFT), specifically LoRA."""
    enabled: bool = False
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = Field(
        default_factory=lambda: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    )
    instruction_tuning: bool = False  # When True, also trains embedding and lm_head layers

class QuantizationConfig(BaseModel):
    """Configuration for model quantization. '4bit' enables QLoRA."""
    mode: Optional[Literal["4bit", "8bit"]] = None

class ModelConfig(BaseModel):
    """Top-level configuration for the model."""
    id: str = "allenai/OLMo-2-1124-7B"
    torch_dtype: str = "auto"
    attn_implementation: Optional[Literal["flash_attention_2"]] = "flash_attention_2"
    peft: PeftConfig = Field(default_factory=PeftConfig)
    quantization: QuantizationConfig = Field(default_factory=QuantizationConfig)

class TrainingConfig(BaseModel):
    """Configuration for the training process, aligned with HF TrainingArguments."""
    output_dir: str = "./results"
    context_length: int = 1024
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 8 # This gives us a effective batch size of 32
    optim: str = "paged_adamw_8bit" # Saves VRAM by using 8bit Adam
    # evaluation_strategy: str = "epoch"
    weight_decay: float = 0.1
    logging_steps: int = 10
    max_grad_norm: float = 0.3
    save_strategy: str = "no" # We'll save manually
    gradient_checkpointing: bool = False # Saves VRAM by using gradient checkpointing
    use_liger_kernel: bool = True # This saves VRAM
        
    # These Hyperparameters are overwritten for LIMA
    num_train_epochs: int = 1
    learning_rate: float = 2e-5
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.03
    seed: int = 42  # For reproducible results

    def to_training_args(self) -> TrainingArguments:
        """Creates a transformers.TrainingArguments object from the config."""
        return TrainingArguments(
            output_dir=self.output_dir,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            num_train_epochs=self.num_train_epochs,
            optim=self.optim,
            weight_decay=self.weight_decay,
            lr_scheduler_type=self.lr_scheduler_type,
            warmup_ratio=self.warmup_ratio,
            # Handle warmup_steps if ratio is not desired (LIMA case)
            warmup_steps=getattr(self, 'warmup_steps', 0),
            logging_steps=self.logging_steps,
            save_strategy=self.save_strategy,
            #evaluation_strategy = self.evaluation_strategy,
            max_grad_norm=self.max_grad_norm,
            report_to="none",
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) and torch.cuda.is_available(),
            use_liger_kernel=self.use_liger_kernel,
            gradient_checkpointing=self.gradient_checkpointing,
            seed=self.seed,
        )
    def to_sft_training_args(self) -> TrainingArguments:
        """Creates a transformers.TrainingArguments object from the config."""
        return SFTConfig(
            dataset_text_field="text",
            padding_free = True, # This saves VRAM (Requires Flash Attention 2)

            # Training Arguments
            output_dir=self.output_dir,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            num_train_epochs=self.num_train_epochs,
            optim=self.optim,
            weight_decay=self.weight_decay,
            lr_scheduler_type=self.lr_scheduler_type,
            warmup_ratio=self.warmup_ratio,
            # Handle warmup_steps if ratio is not desired (LIMA case)
            warmup_steps=getattr(self, 'warmup_steps', 0),
            logging_steps=self.logging_steps,
            save_strategy=self.save_strategy,
            #evaluation_strategy=self.evaluation_strategy,
            max_grad_norm=self.max_grad_norm,
            report_to="none",
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) and torch.cuda.is_available(),
            gradient_checkpointing=self.gradient_checkpointing,
            use_liger_kernel=self.use_liger_kernel,
            seed=self.seed,
        )

class InferenceConfig(BaseModel):
    """Configuration for the inference process."""
    max_new_tokens: int = 512
    temperature: float = 0.1
    top_p: float = 0.95
    repetition_penalty: float = 1.05
    no_repeat_ngram_size: int = 0

# --------------------------------------------------------------------------
# SECTION 2: CORE LLM OPERATIONS
# --------------------------------------------------------------------------

def load_model_for_training(config: ModelConfig, unsloth=False, add_special_token = None):
    """
    Loads a model and tokenizer for training, applying quantization and PEFT.
    **ENHANCED** with robust QLoRA setup from open-instruct.
    """
    log.info(f"Loading model '{config.id}' for training...")

    # Determine torch dtype
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    quant_config = None
    if config.quantization.mode == "4bit":
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype, # Use bfloat16 for compute
            bnb_4bit_use_double_quant=True,
        )
    elif config.quantization.mode == "8bit":
        quant_config = BitsAndBytesConfig(load_in_8bit=True)
    if unsloth:
        pass
        # # Load model
        # model, tokenizer = FastLanguageModel.from_pretrained(
        #     model_name=config.id,
        #     max_seq_length=config.context_length,
        #     dtype=None,  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
        #     full_finetuning = False if config.peft.enabled else True,
        #     # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
        # )
        # if config.peft.enabled:
        #     # Do model patching and add fast LoRA weights
        #     model = FastLanguageModel.get_peft_model(
        #         model,
        #         r=config.peft.lora_r,  # Use config parameter
        #         target_modules=config.peft.target_modules,  # Use config parameter
        #         lora_alpha=config.peft.lora_alpha,  # Use config parameter
        #         lora_dropout=config.peft.lora_dropout,  # Use config parameter
        #         bias="none",  # Bias = "none" is currently optimized
        #         use_gradient_checkpointing=True,
        #         random_state=3407,
        #     )
    else:
        if quant_config == None:
            model = AutoModelForCausalLM.from_pretrained(
                config.id,
                trust_remote_code=True,
                torch_dtype=dtype,
                device_map="auto",
                attn_implementation=config.attn_implementation,
            )
        else:
            print("...Quantizing...")
            model = AutoModelForCausalLM.from_pretrained(
            config.id,
            trust_remote_code=True,
            torch_dtype=dtype,
            quantization_config=quant_config,
            device_map="auto",
            attn_implementation=config.attn_implementation,
        )
        tokenizer = AutoTokenizer.from_pretrained(config.id, trust_remote_code=True)

        # Add special tokens before doing PEFT
        
        if add_special_token is not None:
            log.info(f"Adding special token: {add_special_token}")
            special_tokens_dict = {'additional_special_tokens': [add_special_token]}
            tokenizer.add_special_tokens(special_tokens_dict)  
            model.resize_token_embeddings(len(tokenizer))
            
        # Crucial step for preparing a quantized model for PEFT training.
        if config.quantization.mode:
            model = prepare_model_for_kbit_training(model)

        if config.peft.enabled:
            log.info("Applying PEFT (LoRA)...")
            # Prepare modules_to_save for instruction tuning
            modules_to_save = ["lm_head", "embed_tokens"] if config.peft.instruction_tuning else None
            
            peft_config = PeftLoraConfig(
                r=config.peft.lora_r,
                lora_alpha=config.peft.lora_alpha,
                lora_dropout=config.peft.lora_dropout,
                target_modules=config.peft.target_modules,
                bias="none",
                task_type="CAUSAL_LM",
                modules_to_save=modules_to_save,  # Add this line
            )
            model = get_peft_model(model, peft_config)
            log.info("LoRA applied. Trainable parameters:")
            model.print_trainable_parameters()

    log.info("Model and tokenizer loaded successfully.")
    return model, tokenizer

# **IMPORTANT** Custom trainer to use 'sum' loss, a best practice for chat models.
class SumLossSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False,  **kwargs):
        """
        Computes loss by summing over the sequence dimension, which weights all
        tokens equally. This can improve performance on instruction-following tasks.
        """
        labels = inputs.pop("labels")
        outputs = model(**inputs, use_cache=False)
        logits = outputs.get("logits")

        # Shift so that tokens < n predict n
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = torch.nn.CrossEntropyLoss(reduction="sum")
        loss = loss_fct(shift_logits.view(-1, self.model.config.vocab_size), shift_labels.view(-1))

        # Normalize by the number of examples and gradient accumulation steps
        loss = loss / self.args.per_device_train_batch_size / self.args.gradient_accumulation_steps

        return (loss, outputs) if return_outputs else loss

def fine_tune_on_text(
    model, tokenizer, text_content: str, train_cfg: TrainingConfig, *, context_length: int = 2048 , tag: str = "finetuning on chapter..." # OLMO context limit is 4096 but that might take too much VRAM
):
    """
    Fine-tunes a model on a given string of text.
    **ENHANCED** to use the SumLossSFTTrainer and standardized TrainingArguments.
    """
    if not text_content or not text_content.strip():
        log.warning(f"[{tag}] Text content is empty. Skipping fine-tuning.")
        return

    log.info(f"Starting SFT for '{tag}'...")
    dataset = Dataset.from_dict({"text": [text_content]})

    # Dynamic gradient accumulation: ensures one optimizer step per text blob
    tokens = tokenizer(text_content, add_special_tokens=False, truncation=False)["input_ids"]
    num_chunks = math.ceil(len(tokens) / train_cfg.context_length) if tokens else 1
    grad_accum_steps = max(1, num_chunks)
    log.info(f"[{tag}] Tokens: {len(tokens)}, Context: {train_cfg.context_length} -> Dynamic Grad Accum Steps: {grad_accum_steps}")

    training_args = train_cfg.to_sft_training_args()

    trainer = SumLossSFTTrainer(
        model=model,
        train_dataset=dataset,
        args=training_args,
    )
    trainer.train()
    log.info(f"SFT complete for '{tag}'.")

def run_sft_training(
    model,  train_dataset: Dataset, train_cfg: TrainingConfig
):
    """
    A generalized function to run SFT on a prepared dataset.
    """
    log.info("Starting SFT training run...")
    # This is now much cleaner and correctly uses the passed config.
    training_args = train_cfg.to_sft_training_args()

    trainer = SumLossSFTTrainer(
        model=model,
        train_dataset=train_dataset,
        args=training_args,
    )
    trainer.train()
    log.info("SFT training complete.")
    
    
@torch.inference_mode()
def generate_text(model, tokenizer, prompt: str, config: InferenceConfig) -> str:
    """Simple inference function using Hugging Face transformers.generate."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=config.max_new_tokens,
        temperature=max(config.temperature, 1e-3),
        top_p=config.top_p,
        do_sample=True,
        repetition_penalty=config.repetition_penalty,
        no_repeat_ngram_size = config.no_repeat_ngram_size
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def save_model(model, tokenizer, save_path: str):
    """
    Saves the model and tokenizer. 
    """ # If LoRA was used, it merges the adapters into the base model for easy deployment.
    os.makedirs(save_path, exist_ok=True)
    # if hasattr(model, "merge_and_unload"):
    #     log.info("Merging LoRA adapters and saving full model...")
    #     model = model.merge_and_unload()
    # else:
    log.info("Saving full model...")

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    log.info(f"Model saved to {save_path}")
    
def prepare_lima_dataset(tokenizer: AutoTokenizer, model: AutoModelForCausalLM):
    """
    Loads the GAIR/lima dataset, adds the EOT special token, and formats
    the conversations into a text format suitable for SFTTrainer.

    Args:
        tokenizer: The tokenizer to modify.
        model: The model to resize embeddings for.

    Returns:
        A tuple of (train_dataset, eval_dataset).
    """
    log.info("Preparing GAIR/lima dataset...")
    EOT_TOKEN = "<|EOT|>"
    # 2. Load the dataset
    dataset = load_dataset("GAIR/lima")
    # The paper uses 1000 for training, 50 for dev. The HF dataset has 1030 train examples.
    # We'll split it accordingly.
    train_dataset = dataset["train"].shuffle(seed=42)
    # train_dataset = full_train_dataset
    log.info(f"{len(train_dataset)} training examples.")

    # 3. Define the formatting function
    def format_lima_conversation(example):
        conversation = example['conversations']
        # Join turns with the EOT token. Add one at the very end.
        formatted_text = f"{EOT_TOKEN}".join(conversation) + tokenizer.eos_token
        return {"text": formatted_text}

    # 4. Apply the formatting
    train_dataset = train_dataset.map(format_lima_conversation, remove_columns=['conversations', 'source'])

    return train_dataset
    

/opt/homebrew/anaconda3/envs/finetuning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'peft'

## Initialize

In [2]:
# === Cell 1: Configuration ===
model_config = ModelConfig(
    id="allenai/OLMo-2-1124-7B",
    peft=PeftConfig(
        enabled=True,
        instruction_tuning=True,  # Enable this for LIMA since we're adding EOT token
    ),
    quantization=QuantizationConfig(mode=None), # Use QLoRA
)

log.info("--- Configuration ---")
print(model_config.model_dump_json(indent=2))

2025-06-12 14:27:04 - INFO - [__main__] - --- Configuration ---


{
  "id": "allenai/OLMo-2-1124-7B",
  "torch_dtype": "auto",
  "attn_implementation": "flash_attention_2",
  "peft": {
    "enabled": true,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "instruction_tuning": true
  },
  "quantization": {
    "mode": null
  }
}
{
  "output_dir": "./results",
  "context_length": 1024,
  "per_device_train_batch_size": 2,
  "gradient_accumulation_steps": 8,
  "optim": "paged_adamw_8bit",
  "weight_decay": 0.1,
  "logging_steps": 10,
  "max_grad_norm": 0.3,
  "save_strategy": "no",
  "gradient_checkpointing": false,
  "use_liger_kernel": true,
  "num_train_epochs": 1,
  "learning_rate": 0.00002,
  "lr_scheduler_type": "cosine",
  "warmup_ratio": 0.03,
  "seed": 42
}


In [3]:
# === Cell 2: Load Model for Training ===
log.info("\n--- Loading Model for Training ---")
model, tokenizer = load_model_for_training(model_config, add_special_token="<|EOT|>")


2025-06-12 14:27:08 - INFO - [__main__] - 
--- Loading Model for Training ---
2025-06-12 14:27:08 - INFO - [__main__] - Loading model 'allenai/OLMo-2-1124-7B' for training...
2025-06-12 14:27:08 - INFO - [accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

2025-06-12 14:27:14 - INFO - [__main__] - Adding special token: <|EOT|>
2025-06-12 14:27:14 - INFO - [__main__] - Applying PEFT (LoRA)...
2025-06-12 14:27:19 - INFO - [__main__] - LoRA applied. Trainable parameters:
2025-06-12 14:27:19 - INFO - [__main__] - Model and tokenizer loaded successfully.


trainable params: 861,462,528 || all params: 8,159,481,856 || trainable%: 10.5578


In [4]:
# --- HERE IS THE METHOD ---
footprint_bytes = model.get_memory_footprint()
footprint_gb = footprint_bytes / 1e9  # Convert bytes to gigabytes
print(f"Model dtype: {model.dtype}")
print(f"\nModel Memory Footprint: {footprint_bytes} bytes")
print(f"Model Memory Footprint: {footprint_gb:.2f} GB")

Model dtype: torch.bfloat16

Model Memory Footprint: 16398917888 bytes
Model Memory Footprint: 16.40 GB


### Inspect the Tokenizer 

In [5]:
def inspect_tokenizer(tokenizer):
    """Print tokenizer configuration."""
    print("=== TOKENIZER CONFIG ===")
    print(f"Model: {tokenizer.name_or_path}")
    print(f"Vocab size: {len(tokenizer)}")
    print(f"BOS token: {repr(tokenizer.bos_token)} (ID: {tokenizer.bos_token_id})")
    print(f"EOS token: {repr(tokenizer.eos_token)} (ID: {tokenizer.eos_token_id})")
    print(f"PAD token: {repr(tokenizer.pad_token)} (ID: {tokenizer.pad_token_id})")
    print(f"UNK token: {repr(tokenizer.unk_token)} (ID: {tokenizer.unk_token_id})")
    print(f"Padding side: {getattr(tokenizer, 'padding_side', 'N/A')}")
    print(f"Chat template: {bool(getattr(tokenizer, 'chat_template', None))}")
    
inspect_tokenizer(tokenizer)

=== TOKENIZER CONFIG ===
Model: allenai/OLMo-2-1124-7B
Vocab size: 100279
BOS token: '<|endoftext|>' (ID: 100257)
EOS token: '<|endoftext|>' (ID: 100257)
PAD token: '<|pad|>' (ID: 100277)
UNK token: '<|endoftext|>' (ID: 100257)
Padding side: right
Chat template: False


In [23]:
def audit_dataset_tokenization(dataset, tokenizer, num_samples=3):
    """Inspect how SFT dataset gets tokenized."""
    print("=== DATASET TOKENIZATION AUDIT ===")
    
    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        text = sample["text"]
        
        print(f"\nSample {i}:")
        print(f"Raw text: {repr(text[:200])}...")
        
        # Simulate SFTTrainer tokenization
        tokenized = tokenizer(text, truncation=True, max_length=1024)
        tokens = tokenized["input_ids"]
        
        print(f"Tokenized length: {len(tokens)}")
        print(f"First 10 tokens: {tokens[:10]}")
        print(f"Last 10 tokens: {tokens[-10:]}")
        
        # Check for special tokens
        special_found = []
        if tokenizer.bos_token_id in tokens:
            special_found.append("BOS")
        if tokenizer.eos_token_id in tokens:
            special_found.append("EOS")
        if tokenizer.pad_token_id in tokens:
            special_found.append("PAD")
        if tokenizer.additional_special_tokens[0] in tokens:
            special_found.append("EOT")
        print(f"Special tokens: {special_found}")
        
        # Decode to verify
        decoded = tokenizer.decode(tokens)
        print(f"Decoded matches original: {text.strip() == decoded.strip()}")

In [14]:
tokenizer.additional_special_tokens

['<|EOT|>']

In [24]:
audit_dataset_tokenization(lima_train_ds, tokenizer)

=== DATASET TOKENIZATION AUDIT ===

Sample 0:
Raw text: 'How do you know if you\'re in a healthy relationship?<|EOT|>It is important to understand that there is no "one size fits all" answer to your question. Every relationship is different, and there is no '...
Tokenized length: 374
First 10 tokens: [4438, 656, 499, 1440, 422, 499, 2351, 304, 264, 9498]
Last 10 tokens: [4860, 11, 1243, 701, 5133, 374, 4762, 9498, 13, 100278]
Special tokens: []
Decoded matches original: True

Sample 1:
Raw text: 'Hitler writes a second book called "mein hobby". Write a chapter about one of the many hobbies Hitler indulges in.<|EOT|>Ich sammle Briefmarken. Kein Briefmarken. Ich sammle nur die Briefmarken von al'...
Tokenized length: 140
First 10 tokens: [20065, 1565, 14238, 264, 2132, 2363, 2663, 330, 2727, 258]
Last 10 tokens: [1167, 27960, 6915, 1344, 1751, 11, 1560, 8969, 30, 100278]
Special tokens: []
Decoded matches original: True

Sample 2:
Raw text: '$A$ and $B$ are $n \\times n$ matrices and $v$

### Playing with the base model

In [16]:
question = """Dear student,

You've asked me the following question: "What is the essence of calculus?"

Let me answer your question. """
generated_text = generate_text(model, tokenizer, question, inference_config)


In [17]:
print(generated_text)

Dear student,

You've asked me the following question: "What is the essence of calculus?"

Let me answer your question.  The essence of calculus is the study of change.  Calculus is the study of how things change.  It is the study of how things change with respect to time, or how things change with respect to other things.  Calculus is the study of how things change.

Calculus is the study of how things change.  It is the study of how things change with respect to time, or how things change with respect to other things.  Calculus is the study of how things change.

Calculus is the study of how things change.  It is the study of how things change with respect to time, or how things change with respect to other things.  Calculus is the study of how things change.

Calculus is the study of how things change.  It is the study of how things change with respect to time, or how things change with respect to other things.  Calculus is the study of how things change.

Calculus is the study of h

In [23]:
question = """Dear student,

You said the following earlier: "I think the Law of Large Numbers also tells us something similar to what the Central Limit Theoreom says."

Let me evaluate your understanding. You"""
generated_text = generate_text(model, tokenizer, question, inference_config)


In [24]:
print(generated_text)

Dear student,

You said the following earlier: "I think the Law of Large Numbers also tells us something similar to what the Central Limit Theoreom says."

Let me evaluate your understanding. You are correct that the law of large numbers tells you that if you take a large number of samples from a population, the sample mean will be close to the population mean. This is true for any population distribution, not just the normal distribution. The central limit theorem tells a different story. It says that even if the underlying population is not normally distributed, if we take enough samples, their means will follow a normal curve. So the central l"

Do not Just list concepts, but develop each one in detail before moving to next, as we prioritize depth of understanding and comprehensive exploration of the subject matter over breadth. Focus on:

- Rigor: Ensure in-depth coverage of concepts/sections.
- Engagement: Write with an academic, professional and engaging tone that captivates inte

## LIMA Alignment

In [5]:
lima_training_config = TrainingConfig(
    output_dir="./results/olmo2_7b_lima_aligned",
    num_train_epochs = 10,
    learning_rate  = 1e-5,
    lr_scheduler_type = "cosine",
    warmup_steps  = 0, # LIMA specifies no warmup, so we set this explicitly
    warmup_ratio = 0.3 
    )
    
inference_config = InferenceConfig()

In [6]:
# === Cell 3: Prepare LIMA Dataset (This modifies the model and tokenizer) ===
log.info("\n--- Preparing LIMA Dataset ---")
# This function adds the EOT token and resizes model embeddings in-place
lima_train_ds = prepare_lima_dataset(tokenizer, model)
log.info(f"Sample formatted training example:\n{lima_train_ds[0]['text']}")


2025-06-12 14:27:20 - INFO - [__main__] - 
--- Preparing LIMA Dataset ---
2025-06-12 14:27:20 - INFO - [__main__] - Preparing GAIR/lima dataset...
2025-06-12 14:27:22 - INFO - [__main__] - 1030 training examples.
2025-06-12 14:27:22 - INFO - [__main__] - Sample formatted training example:
How do you know if you're in a healthy relationship?<|EOT|>It is important to understand that there is no "one size fits all" answer to your question. Every relationship is different, and there is no single way to define a "healthy" relationship.

That said, there are some general guidelines that you can use. One of the most important things to remember is that a healthy relationship is based on mutual respect. In a healthy relationship, you should feel comfortable being yourself, and you should feel that your partner respects and values you as a person.

Another important aspect of a healthy relationship is honesty. In a healthy relationship, you should feel comfortable being open and honest with you

In [7]:
# === Cell 4: Run LIMA Fine-Tuning ===
log.info("\n--- Starting LIMA Fine-Tuning ---")
# The model object will be updated with the fine-tuned weights
run_sft_training(
    model=model,
    train_dataset=lima_train_ds,
    train_cfg=lima_training_config,
)

2025-06-12 14:27:22 - INFO - [__main__] - 
--- Starting LIMA Fine-Tuning ---
2025-06-12 14:27:22 - INFO - [__main__] - Starting SFT training run...
2025-06-12 14:27:23 - INFO - [liger_kernel.transformers.monkey_patch] - Applying Liger kernels to model instance with model type: olmo2 with kwargs: {}
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,1059.221600
20,992.697200
30,1064.032700
40,1016.723600
50,946.742700
60,1001.401400
70,1021.534400
80,1073.696700
90,988.009500
100,972.805400


2025-06-12 15:22:08 - INFO - [__main__] - SFT training complete.


In [13]:
# git config --global user.email "jiosephlee@gmail.com"
# git config --global user.name "Joseph Lee"
log.info("\n--- Running Inference with LIMA-aligned Model ---")
EOT_TOKEN = "<|EOT|>"
inference_config = InferenceConfig(no_repeat_ngram_size=6)
question = f"""Can you explain the essence of calculus to me?{EOT_TOKEN}"""
generated_text = generate_text(model, tokenizer, question, inference_config)
print(generated_text)

2025-06-12 15:28:32 - INFO - [__main__] - 
--- Running Inference with LIMA-aligned Model ---


Can you explain the essence of calculus to me?<|EOT|>Calculus is the study of change. It is the mathematics of motion, of growth, of rates of change. It is a tool for understanding the world around us. It is the mathematics that underlies all of physics, all of engineering, all of economics, all of biology, all of medicine, all of computer science, and all of statistics. It is the mathematics behind the design of the computer you are using to read this, the mathematics behind the algorithms that make it work, the mathematics behind the software that runs on it, the mathematics behind the networks that connect it to the rest of the world, the mathematics behind the design and construction of the buildings that house the computers, the mathematics behind the cars that transport us to and from those buildings, the mathematics behind the airplanes that transport us across the world, the mathematics that underlies the design of the buildings that house us, the mathematics behind the medicin

In [14]:
len(generated_text)

1682

In [15]:
# === Cell 5: Test the Aligned Model ===
log.info("\n--- Running Inference with LIMA-aligned Model ---")
# The prompt must now follow the LIMA format, ending with the EOT token
# to signal that it's the assistant's turn to speak.
EOT_TOKEN = "<|EOT|>"
prompt = f"Can you explain the theory of relativity in simple terms?{EOT_TOKEN}"

generated_text = generate_text(model, tokenizer, prompt, inference_config)

print("\n" + "="*50)
print(" " * 15 + "INFERENCE RESULT")
print("="*50)
print(f"PROMPT:\n{prompt}\n")
print(f"GENERATED:\n{generated_text}")
print("="*50 + "\n")

# The model should have learned to stop generating at the EOT token.

2025-06-12 15:29:32 - INFO - [__main__] - 
--- Running Inference with LIMA-aligned Model ---



               INFERENCE RESULT
PROMPT:
Can you explain the theory of relativity in simple terms?<|EOT|>

GENERATED:
Can you explain the theory of relativity in simple terms?<|EOT|>Relativity is a theory that describes the relationship between space and time. It was developed by Albert Einstein in the early 1900s. The theory has two main parts: special relativity and general relativity.

Special relativity is a theory that applies to objects that are moving at a constant speed in a straight line. It states that the laws of physics are the same for all observers, regardless of their speed or direction of motion. This means that if you were to measure the speed of light in a vacuum, you would always get the same result, no matter how fast you were moving.

General relativity is a theory of gravity. It states that gravity is not a force, but rather a curvature of spacetime. This means that massive objects like planets and stars bend the fabric of spacetime around them, causing other obje

## Now teaching the model via fine-tuning for each chapter

In [1]:
def fine_tune_on_text(
    model, tokenizer, text_content: str, train_cfg: TrainingConfig, 
    *, tag: str = "finetuning on text..."
):
    """
    Fine-tunes a model on a given string of text by chunking it properly.
    
    Args:
        model: The model to fine-tune
        tokenizer: The tokenizer
        text_content: The text to fine-tune on
        train_cfg: Training configuration
        context_length: Override context length (uses train_cfg.context_length if None)
        tag: Tag for logging
    """
    if not text_content or not text_content.strip():
        log.warning(f"[{tag}] Text content is empty. Skipping fine-tuning.")
        return

    log.info(f"Starting SFT for '{tag}'...")
    
    # Tokenize the full text without truncation
    tokens = tokenizer(text_content, add_special_tokens=False, truncation=False)["input_ids"]
    num_tokens = len(tokens)
    
    if num_tokens == 0:
        log.warning(f"[{tag}] Text tokenized to 0 tokens. Skipping fine-tuning.")
        return
    context_length = train_cfg.context_length
    
    # Calculate number of chunks
    num_chunks = math.ceil(num_tokens / context_length)
    log.info(f"[{tag}] Tokens: {num_tokens}, Context: {context_length} -> {num_chunks} chunks")
    
    # Chunk the tokens and decode back to text
    text_chunks = []
    for i in range(num_chunks):
        start_idx = i * context_length
        end_idx = min((i + 1) * context_length, num_tokens)
        chunk_tokens = tokens[start_idx:end_idx]
        
        # Decode chunk back to text
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=False)
        text_chunks.append(chunk_text)
    
    # Create dataset with chunked text
    dataset = Dataset.from_dict({"text": text_chunks})
    log.info(f"[{tag}] Created dataset with {len(text_chunks)} chunks")
    
    # Create a modified training config with dynamic gradient accumulation
    # This ensures one optimizer step after processing all chunks
    train_cfg.gradient_accumulation_steps = num_chunks
    
    log.info(f"[{tag}] Setting gradient_accumulation_steps to {num_chunks} (one optimizer step per document)")
    
    training_args = train_cfg.to_sft_training_args()
    
    trainer = SumLossSFTTrainer(
        model=model,
        train_dataset=dataset,
        args=training_args,
    )
    
    trainer.train()
    log.info(f"SFT complete for '{tag}'.")

NameError: name 'TrainingConfig' is not defined

In [ ]:
training_config = TrainingConfig(
    context_length=2048,
    num_train_epochs=10,
    learning_rate=1e-5,
)

### Some areas we want to examine
- Academic Concepts (Both STEM and NOT-STEM; Does it increase its understanding? STEM is unique because it especially includes mathematical notation which is more out of context For Non-STEM, I'd say it's the most in-distribution because it's pure english like English or Law but we'd have to check the pre-training data on that one)
- Policy white-papers (Can it make better arguments as well?)
- Summarization of short stories 
- Re-argugin books that made philosophical arguments; Chapters
- Continuing the story with open-ended endings (Creation; Creativity, Fullness, How well it ties to the original story)
- Reasoning; Problem Solving (explaining how to write DP problems, doing physics problems with tension, integrating by parts)
- In-context learning (with chess puzzles)
- Recent Academic Papers (Novel and Out of distribution)
https://chatgpt.com/share/684bf677-cc6c-8011-ae20-b049eba7f65f 

Textbook 

In [ ]:
question= "What is the intuition behind the gram-schmidt procedure?" + EOT_TOKEN

passage= """Given a vector $v$ and some other vector $u$ not in ${\\rm span}\\, \\{v\\}$, we can construct a new vector: $v^{\\perp}:=v-\\dfrac{u\\cdot v}{u\\cdot u}\\, u\\, .$\n\nThis new vector $v^{\\perp}$ is orthogonal to $u$ because\n\n﻿$u\\cdot v^{\\perp} = u\\cdot v - \\dfrac{u\\cdot v}{u\\cdot u}u\\cdot u = 0.$\n\nHence, $\\{u, v^{\\perp}\\}$ is an orthogonal basis for $span \\{u,v\\}$. When $v$ is not parallel to $u$, $v^{\\perp} \\neq 0$, and normalizing these vectors we obtain $\\{\\dfrac{u}{|u|}, \\dfrac{v^{\\perp}}{|v^{\\perp}|}\\}$, an orthonormal basis for the vector space ${\\rm span}\\, \\{u,v\\}$.\n\nSometimes we write $v = v^{\\perp} + v^{\\parallel}$ where:\n\n\\begin{eqnarray*}\nv^{\\perp} &=& v-\\dfrac{u\\cdot v}{u\\cdot u}u \\\nv^{\\parallel} &=& \\phantom{v-}\\dfrac{u\\cdot v}{u\\cdot u}u.\n\\end{eqnarray*}\n\nThis is called an $\\textit{orthogonal decomposition}$ because we have decomposed $v$ into a sum of orthogonal vectors. This decomposition depends on $u$; if we change the direction of $u$ we change $v^{\\perp}$ and $v^{\\parallel}$.\n\nIf $u$, $v$ are linearly independent vectors in $\\Re^{3}$, then the set $\\{u, v^{\\perp}, u\\times v^{\\perp} \\}$ would be an orthogonal basis for $\\Re^{3}$. This set could then be normalized by dividing each vector by its length to obtain an orthonormal basis.\n\nHowever, it often occurs that we are interested in vector spaces with dimension greater than $3$, and must resort to craftier means than cross products to obtain an orthogonal basis.﻿\n\nGiven a third vector $w$, we should first check that $w$ does not lie in the span of $u$ and $v$, $\\textit{i.e.}$, check that $u,v$ and $w$ are linearly independent. If it does not, we then can define:\n\n$w^{\\perp} = w - \\dfrac{u\\cdot w}{u\\cdot u}\\,u - \\dfrac{v^{\\perp}\\cdot w}{v^{\\perp}\\cdot v^{\\perp}}\\,v^{\\perp}.$\n\nWe can check that $u \\cdot w^{\\perp}$ and $v^{\\perp} \\cdot w^{\\perp}$ are both zero:\n\n\\begin{align*}\nu \\cdot w^{\\perp}&=u \\cdot \\left(w - \\dfrac{u\\cdot w}{u\\cdot u}\\,u - \\dfrac{v^{\\perp}\\cdot w}{v^{\\perp}\\cdot v^{\\perp}}\\,v^{\\perp} \\right)\\&= u\\cdot w - \\dfrac{u \\cdot w}{u \\cdot u}u \\cdot u - \\dfrac{v^{\\perp} \\cdot w}{v^{\\perp} \\cdot v^{\\perp}} u \\cdot v^{\\perp} \\\n&=u\\cdot w-u\\cdot w-\\dfrac{v^{\\perp} \\cdot w}{v^{\\perp} \\cdot v^{\\perp}} u \\cdot v^{\\perp}\\ =\\ 0\n\\end{align*}\n\nsince $u$ is orthogonal to $v^{\\perp}$, and\n\n\\begin{align*}\nv^{\\perp} \\cdot w^{\\perp}&=v^{\\perp} \\cdot \\left(w - \\dfrac{u\\cdot w}{u\\cdot u}\\,u - \\dfrac{v^{\\perp}\\cdot w}{v^{\\perp}\\cdot v^{\\perp}}\\,v^{\\perp} \\right)\\ &=v^{\\perp}\\cdot w - \\dfrac{u \\cdot w}{u \\cdot u}v^{\\perp} \\cdot u - \\dfrac{v^{\\perp} \\cdot w}{v^{\\perp} \\cdot v^{\\perp}} v^{\\perp} \\cdot v^{\\perp} \\\n&=v^{\\perp}\\cdot w-\\dfrac{u \\cdot w}{u \\cdot u}v^{\\perp} \\cdot u - v^{\\perp} \\cdot w\\ =\\ 0\n\\end{align*}\n\nbecause $u$ is orthogonal to $v^{\\perp}$. Since $w^{\\perp}$ is orthogonal to both $u$ and $v^{\\perp}$, we have that $\\{u,v^{\\perp},w^{\\perp} \\}$ is an orthogonal basis for $span \\{u,v,w\\}$.\n\nThe Gram-Schmidt Procedure\n\nIn fact, given a set $\\{v_{1}, v_{2}, \\ldots \\}$ of linearly independent vectors, we can define an orthogonal basis for $span \\{v_{1},v_{2}, \\ldots \\}$ consisting of the following vectors:\n\n\\begin{eqnarray*}\nv_{1}^{\\perp}&:=&v_{1} \\\nv_{2}^{\\perp} &:=& v_{2} - \\dfrac{v_{1}^{\\perp}\\cdot v_{2}}{v_{1}^{\\perp}\\cdot v_{1}^{\\perp}}\\,v_{1}^{\\perp} \\\nv_{3}^{\\perp} &:=& v_{3} - \\dfrac{v_{1}^{\\perp}\\cdot v_{3}}{v_{1}^{\\perp}\\cdot v_{1}^{\\perp}}\\,v_{1}^{\\perp} - \\dfrac{v_{2}^{\\perp}\\cdot v_{3}}{v_{2}^{\\perp}\\cdot v_{2}^{\\perp}}\\,v_{2}^{\\perp}\\\n&\\vdots& \\\nv_{i}^{\\perp} &=& v_{i} - \\sum_{j<i} \\dfrac{v_{j}^{\\perp}\\cdot v_{i}}{v_{j}^{\\perp}\\cdot v_{j}^{\\perp}}\\,v_{j}^{\\perp} \\\n&:=& v_{i} - \\dfrac{v_{1}^{\\perp}\\cdot v_{i}}{v_{1}^{\\perp}\\cdot v_{1}^{\\perp}}\\,v_{1}^{\\perp}\n- \\dfrac{v_{2}^{\\perp}\\cdot v_{i}}{v_{2}^{\\perp}\\cdot v_{2}^{\\perp}}\\,v_{2}^{\\perp} -\\cdots\n- \\dfrac{v_{i-1}^{\\perp}\\cdot v_{i}}{v_{i-1}^{\\perp}\\cdot v_{i-1}^{\\perp}}\\,v_{i-1}^{\\perp}\\\n&\\vdots& \\\n\\end{eqnarray*}\n\nNotice that each $v_{i}^{\\perp}$ here depends on $v_{j}^{\\perp}$ for every $j<i$. This allows us to inductively/algorithmically build up a linearly independent, orthogonal set of vectors $\\{v_{1}^{\\perp},v_{2}^{\\perp}, \\ldots \\}$ such that $span \\{v_{1}^{\\perp},v_{2}^{\\perp}, \\ldots \\}=span \\{v_{1}, v_{2}, \\ldots \\}$. That is, an orthogonal basis for the latter vector space. This algorithm is called the $\\textit{Gram--Schmidt orthogonalization procedure}$--Gram worked at a Danish insurance company over one hundred years ago, Schmidt was a student of Hilbert (the famous German mathmatician).\n\nExample $1$:\n\nWe'll obtain an orthogonal basis for $\\Re^{3}$ by appling Gram-Schmidt to the linearly independent set $\\{ v_{1}=\\begin{pmatrix}1\\1\\0\\end{pmatrix}, v_{2}=\\begin{pmatrix}1\\1\\1\\end{pmatrix},v_{3}=\\begin{pmatrix}3\\1\\1\\end{pmatrix}\\}$. First, we set $v_{1}^{\\perp}:=v_{1}$. Then:\n\n\\begin{eqnarray*}\nv_{2}^{\\perp}&=& \\begin{pmatrix}1\\1\\1\\end{pmatrix} - \\dfrac{2}{2}\\begin{pmatrix}1\\1\\0\\end{pmatrix} = \\begin{pmatrix}0\\0\\1\\end{pmatrix} \\\nv_{3}^{\\perp}&=& \\begin{pmatrix}3\\1\\1\\end{pmatrix} - \\dfrac{4}{2}\\begin{pmatrix}1\\1\\0\\end{pmatrix} - \\dfrac{1}{1}\\begin{pmatrix}0\\0\\1\\end{pmatrix} = \\begin{pmatrix}1\\-1\\0\\end{pmatrix}.\n\\end{eqnarray*}\n\nThen the set\n\n$\\left\\{ \\begin{pmatrix}1\\1\\0\\end{pmatrix},\\begin{pmatrix}0\\0\\1\\end{pmatrix},\\begin{pmatrix}1\\-1\\0\\end{pmatrix}\\right\\}$\n\nis an orthogonal basis for $\\Re^{3}$. To obtain an orthonormal basis, as always we simply divide each of these vectors by its length, yielding:\n\n$\\left\\{ \\begin{pmatrix}\\dfrac{1}{\\sqrt2}\\\\dfrac{1}{\\sqrt2}\\0\\end{pmatrix},\\begin{pmatrix}0\\0\\1\\end{pmatrix},\\begin{pmatrix}\\dfrac{1}{\\sqrt2}\\\\dfrac{-1}{\\sqrt2}\\0\\end{pmatrix}\\right\\}.$\n\n﻿﻿﻿﻿\n\n14.05: QR Decomposition\n\nIn chapter 7, section 7.7 we learned how to solve linear systems by decomposing a matrix $M$ into a product of lower and upper triangular matrices\n\n$M=LU\\, .$\n\nThe Gram-Schmidt procedure suggests another matrix decomposition,\n\n$M=QR\\, ,$\n\nwhere $Q$ is an orthogonal matrix and $R$ is an upper triangular matrix. So-called QR-decompositions are useful for solving linear systems, eigenvalue problems and least squares approximations. You can easily get the idea behind the $QR$ decomposition by working through a simple example.\n\nExample $1$:\n\nFind the $QR$ decomposition of\n\n$M=\\begin{pmatrix}2&-1&1\\1&3&-2\\0&1&-2\\end{pmatrix}\\, .$\n\nWhat we will do is to think of the columns of $M$ as three 3-vectors and use Gram-Schmidt to build an orthonormal basis from these that will become the columns of the orthogonal matrix $Q$. We will use the matrix $R$ to record the steps of the Gram-Schmidt procedure in such a way that the product $QR$ equals $M$.\n\nTo begin with we write\n\n$M=\\begin{pmatrix}2&-\\frac{7}{5}&1\\1&\\frac{14}{5}&-2\\0&1&-2\\end{pmatrix} \\begin{pmatrix}1&\\frac{1}{5}&0\\0&1&0\\0&0&1\\end{pmatrix}\\, .$\n\nIn the first matrix the first two columns are orthogonal because we simply replaced the second column of $M$ by the vector that the Gram-Schmidt procedure produces from the first two columns of $M$, namely\n\n$\\begin{pmatrix}-\\frac{7}{5}\\\\frac{14}{5}\\1\\end{pmatrix}=\\begin{pmatrix}-1\\3\\1\\end{pmatrix}-\\frac{1}{5} \\begin{pmatrix} 2 \\1\\0\\end{pmatrix}\\, .$\n\nThe matrix on the right is almost the identity matrix, save the $+\\frac{1}{5}$ in the second entry of the first row, whose effect upon multiplying the two matrices precisely undoes what we we did to the second column of the first matrix. For the third column of $M$ we use Gram--Schmidt to deduce the third orthogonal vector\n\n$\\begin{pmatrix}-\\frac{1}{6}\\\\frac{1}{3}\\-\\frac{7}{6}\\end{pmatrix}=\\begin{pmatrix}1\\-2\\-2\\end{pmatrix}-0\\begin{pmatrix}2\\1\\0\\end{pmatrix}-\\frac{-9}{\\frac{54}{5}}\\begin{pmatrix}-\\frac{7}{5}\\\\frac{14}{5}\\1\\end{pmatrix}\\,$\n\nand therefore, using exactly the same procedure write\n\n$M=\\begin{pmatrix}2&-\\frac{7}{5}&-\\frac{1}{6}\\1&\\frac{14}{5}&\\frac{1}{3}\\0&1&-\\frac{7}{6}\\end{pmatrix} \\begin{pmatrix}1&\\frac{1}{5}&0\\0&1&-\\frac{5}{6}\\0&0&1\\end{pmatrix}\\, .$\n\nThis is not quite the answer because the first matrix is now made of mutually orthogonal column vectors, but a $\\textit{bona fide}$ orthogonal matrix is comprised of orthonormal vectors. To achieve that we divide each column of the first matrix by its length and multiply the corresponding row of the second matrix by the same amount:\n\n$M=\\begin{pmatrix}\\frac{2\\sqrt{5}}{5}&-\\frac{7\\sqrt{30}}{90}&-\\frac{\\sqrt{6}}{18}\\ \\frac{\\sqrt{5}}{5}&\\frac{7\\sqrt{30}}{45}&\\frac{\\sqrt{6}}{9}\\ 0&\\frac{\\sqrt{30}}{18}&-\\frac{7\\sqrt{6}}{18}\\end{pmatrix} \\begin{pmatrix}\\sqrt{5}&\\frac{\\sqrt{5}}{5}&0\\ 0&\\frac{3\\sqrt{30}}{5}&-\\frac{\\sqrt{30}}{2}\\ 0&0&\\frac{\\sqrt{6}}{2}\\end{pmatrix}=QR\\, .$\n\nA nice check of this result is to verify that entry $(i,j)$ of the matrix $R$ equals the dot product of the $i$-th column of $Q$ with the $j$-th column of $M$. (Some people memorize this fact and use it as a recipe for computing $QR$ deompositions.) $\\textit{A good test of your own understanding is to work out why this is true!}$",
"key_points": "- When you project a vector onto a subspace (any dimension), and subtract the result, you get a vector orthogonal to the subspace\n- Projection onto an n dimensional subspace, on the other hand, can be accomplished by adding the projections onto the n individual members of an orthogonal basis.""" + tokenizer.eos_token

In [ ]:

# === Cell 3: Prepare Text and Fine-Tune ===
log.info("\n--- Fine-Tuning on Custom Text ---")
fine_tune_on_text(
    model=model,
    tokenizer=tokenizer,
    text_content=passage,
    train_cfg=training_config,
)

# === Cell 4: Run Inference with the Fine-Tuned Model ===
log.info("\n--- Running Inference ---")
generated_text = generate_text(model, tokenizer, question, inference_config)

print("\n" + "="*50)
print(" " * 15 + "INFERENCE RESULT")
print("="*50)
print(generated_text)
print("="*50 + "\n")


In [16]:
# === Cell 5: Save the Final Model ===
log.info("\n--- Saving Final Model ---")
final_model_path = "./results/olmo2_7b_ml_chapter"
save_model(model, tokenizer, final_model_path)
log.info(f"Final merged model ready for deployment at {final_model_path}")

2025-06-12 15:32:25 - INFO - [__main__] - 
--- Saving Final Model ---
2025-06-12 15:32:25 - INFO - [__main__] - Saving full model...
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
2025-06-12 15:32:33 - INFO - [__main__] - Model saved to ./results/olmo2_7b_lima
2025-06-12 15:32:33 - INFO - [__main__] - Final merged model ready for deployment at ./results/olmo2_7b_lima


## Merge with LIMA Adapter

Load the LIMA Adapter (on hugginface hug jiosephlee/olmo2-lima) with the current adapter and merge via DARE

In [ ]:
model # Already loaded with an adapter
_ = model.load_adapter("jiosephlee/olmo-lima", adapter_name="lima")

adapters = ["chapter", "lima"]
weights = [1.0, 1.0]
adapter_name = "merge"
density = 0.2
model.add_weighted_adapter(adapters, weights, adapter_name, combination_type="dare_ties", density=density)

model.set_adapter("merge")
merged_model = model.merge_and_unload()

## Save model

In [18]:
model.push_to_hub('jiosephlee/olmo2-lima')

adapter_model.safetensors:   0%|          | 0.00/5.09G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/jiosephlee/olmo2-lima/commit/3f09aeb24a545c02179958a588eacb29a18b3c06', commit_message='Upload model', commit_description='', oid='3f09aeb24a545c02179958a588eacb29a18b3c06', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jiosephlee/olmo2-lima', endpoint='https://huggingface.co', repo_type='model', repo_id='jiosephlee/olmo2-lima'), pr_revision=None, pr_num=None)

In [19]:
tokenizer.push_to_hub('jiosephlee/olmo2-lima')

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/jiosephlee/olmo2-lima/commit/5ef15ba0cb06170025e886b07dc0712e1db9783c', commit_message='Upload tokenizer', commit_description='', oid='5ef15ba0cb06170025e886b07dc0712e1db9783c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jiosephlee/olmo2-lima', endpoint='https://huggingface.co', repo_type='model', repo_id='jiosephlee/olmo2-lima'), pr_revision=None, pr_num=None)